# Does rain lower turnout? The notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/john-bruene/pointzerofive-data/blob/main/rain-lowers-turnout/story_007_rain.ipynb)

This R notebook recomputes every statistic in
[Does rain lower turnout?](https://pointzerofive.net/2026/10/rain-lowers-turnout), story #007 on
*pointzerofive*, from two small tables: turnout per Wahlkreis at the Bundestag elections 2005–2025
and the mean rain per Wahlkreis on the election dates of every year 1995–2025. Colab opens it with
the R runtime; run it top to bottom (`Runtime → Run all`). The seeds are the pipeline's, so every
number matches exactly.

In [ ]:
library(jsonlite)

BASE <- "https://raw.githubusercontent.com/john-bruene/pointzerofive-data/main/rain-lowers-turnout/"
REPO <- ".."
fetch_path <- function(rel) {
  for (local in c(basename(rel), file.path(REPO, rel))) if (file.exists(local)) return(local)
  paste0(BASE, basename(rel))
}
turnout <- read.csv(fetch_path("notebooks/raw/rain/turnout.csv"), stringsAsFactors = FALSE)
rain <- read.csv(fetch_path("notebooks/raw/rain/rain.csv"), stringsAsFactors = FALSE)
published <- fromJSON(fetch_path("data/rain-lowers-turnout.json"), simplifyVector = FALSE)
c(turnout = nrow(turnout), rain = nrow(rain))

## 1. Turnout change and rain change

y = change in turnout from the previous election in percentage points (the previous result is
already on the current boundaries). Rain = mean HYRAS precipitation over the Wahlkreis; the value
of a date covers 06 UTC that day to 06 UTC the next. Left out, as fixed in the pitch: Dresden I in
2005 and the Berlin Wahlkreise in 2025.

In [ ]:
YEARS <- 1995:2025
ELECTIONS <- c(2002L, 2005L, 2009L, 2013L, 2017L, 2021L, 2025L)
t <- turnout
t$y <- 100 * (t$voters / t$eligible - t$voters_prev / t$eligible_prev)
t <- t[!((t$election == 2005 & t$name == "Dresden I") | (t$election == 2025 & t$land == 11)), ]
mat <- function(which) {
  s <- rain[rain$which == which, ]
  full <- tapply(s$mm, list(paste(s$geometry, s$which, s$wkr), s$year), identity)
  unname(full[paste(t$election, which, t$wkr), as.character(YEARS), drop = FALSE])
}
this <- mat("this"); prev <- mat("prev")
date_this <- match(t$election, ELECTIONS); date_prev <- date_this - 1L
t$rain_this <- this[cbind(seq_len(nrow(t)), match(t$election, YEARS))]
t$rain_prev <- prev[cbind(seq_len(nrow(t)), match(ELECTIONS[date_prev], YEARS))]
t$clim <- rowMeans(this) - rowMeans(prev)
stopifnot(nrow(t) == published$meta$observations)
# Fig. 1: rain on each election day
for (e in published$elections) {
  mine <- if (e$election == 2002) t$rain_prev[t$election == 2005] else t$rain_this[t$election == e$election]
  stopifnot(max(abs(sort(mine) - unlist(e$rain))) < 0.006)
  cat(sprintf("%d: median %.1f mm, wettest Wahlkreis %.1f mm, %2.0f%% with >= 1 mm\n",
              e$election, median(mine), max(mine), 100 * mean(mine >= 1)))
}

## 2. The slope and the placebo test

The effect is the least-squares slope of y on the rain change (per 10 mm) with one constant per
group. The placebo replaces the rain field of every election date by the field of the same date
in another year 1995–2025 (one draw per date and replicate). Test of β = b: |β̂ − b| against
|a − b·c|, with a the placebo slope of y and c the placebo slope of the observed rain change.
p = p(0); the 95% interval is every b with p(b) ≥ 0.05.

In [ ]:
within_slope <- function(y, X, g, w) {
  X <- as.matrix(X)
  sw <- rowsum(w, g)
  xm <- rowsum(w * X, g) / as.vector(sw); ym <- rowsum(w * y, g) / as.vector(sw)
  gi <- match(g, rownames(sw))
  Xt <- X - xm[gi, , drop = FALSE]; yt <- y - ym[gi]
  list(slope = colSums(w * Xt * yt) / colSums(w * Xt^2), Xt = Xt)
}
# check against lm() with dummies for the main calculation
x0 <- (t$rain_this - t$rain_prev) / 10
g0 <- paste(t$election, t$land)
stopifnot(abs(within_slope(t$y, x0, g0, rep(1, nrow(t)))$slope - coef(lm(t$y ~ x0 + factor(g0)))[["x0"]]) < 1e-10)

invert <- function(p_of, beta, s) {
  grid <- seq(beta - 10 * s, beta + 10 * s, length.out = 4001)
  ok <- vapply(grid, function(b) p_of(b) >= 0.05, TRUE)
  inside <- which(ok)
  edge <- function(out, inn) { for (k in 1:40) { mid <- (out + inn) / 2; if (p_of(mid) >= 0.05) inn <- mid else out <- mid }; inn }
  c(edge(grid[min(inside) - 1], grid[min(inside)]), edge(grid[max(inside) + 1], grid[max(inside)]))
}
test_path <- function(rain_m, within, elections, weight, R = 10000L) {
  i <- if (elections == "pre2021") which(t$election <= 2017) else seq_len(nrow(t))
  s <- t[i, ]
  x <- (s$rain_this - s$rain_prev - if (rain_m == "anomaly") s$clim else 0) / 10
  g <- if (within == "land") paste(s$election, s$land) else as.character(s$election)
  w <- if (weight == "voters") s$eligible / mean(s$eligible) else rep(1, nrow(s))
  beta <- within_slope(s$y, x, g, w)$slope
  xt <- within_slope(x, x, g, w)$Xt[, 1]
  a <- cc <- numeric(R)
  for (start in seq(1L, R, by = 500L)) {
    m <- min(500L, R - start + 1L)
    yrs <- matrix(vapply(seq_along(ELECTIONS), function(k)
      sample(setdiff(seq_along(YEARS), match(ELECTIONS[k], YEARS)), m, replace = TRUE), integer(m)), nrow = m)
    Xp <- vapply(seq_len(m), function(j)
      this[cbind(i, yrs[j, date_this[i]])] - prev[cbind(i, yrs[j, date_prev[i]])], numeric(length(i)))
    Xp <- (Xp - if (rain_m == "anomaly") s$clim else 0) / 10
    fit <- within_slope(s$y, Xp, g, w)
    a[start:(start + m - 1)] <- fit$slope
    cc[start:(start + m - 1)] <- colSums(w * fit$Xt * xt) / colSums(w * fit$Xt^2)
  }
  p_of <- function(b) mean(abs(a - b * cc) >= abs(beta - b))
  ci <- invert(p_of, beta, sd(a))
  list(effect = beta, ci_low = ci[1], ci_high = ci[2], p = p_of(0), n = length(i), placebo = a)
}

## 3. All 16 calculation paths

Seed 7100 + path number, forks in the order of the pitch.

In [ ]:
rows <- list(); k <- 0; main <- NULL
for (rain_m in c("day", "anomaly")) for (within in c("land", "election"))
  for (elections in c("all", "pre2021")) for (weight in c("none", "voters")) {
    k <- k + 1
    set.seed(7100L + k)
    r <- test_path(rain_m, within, elections, weight)
    if (k == 1) main <- r
    rows[[k]] <- data.frame(rain = rain_m, within, elections, weight, n = r$n, effect = r$effect,
                            ci_low = r$ci_low, ci_high = r$ci_high, p = r$p)
  }
specs <- do.call(rbind, rows)
for (k in seq_len(nrow(specs))) {
  pub <- published$specs[[k]]
  stopifnot(identical(unname(unlist(pub$choices)), unname(unlist(specs[k, 1:4]))), pub$n == specs$n[k])
  for (col in c("effect", "ci_low", "ci_high", "p")) stopifnot(abs(pub[[col]] - specs[[col]][k]) < 1e-9)
}
print(specs, digits = 3)

## 4. The verdict

Main calculation: rain on the day, constants per Land and election, all elections, every Wahlkreis
once. "Yes" needs p < 0.05 and a negative effect. Smallest relevant effect: 0.4 percentage points
per 10 mm.

In [ ]:
m <- specs[1, ]
verdict <- if (m$p < 0.05 && m$effect < 0) "yes" else "no"
cat(sprintf("main: %+.3f pp per 10 mm, 95%% CI [%+.3f, %+.3f], p = %.4f, n = %d -> %s\n",
            m$effect, m$ci_low, m$ci_high, m$p, m$n, verdict))
cat(sprintf("a relevant negative effect (<= -0.4) is %s\n", if (m$ci_low > -0.4) "ruled out" else "not ruled out"))
cat(sprintf("yes on %d of %d paths\n", sum(specs$p < 0.05 & specs$effect < 0), nrow(specs)))
stopifnot(verdict == "no")
pl <- published$placebo
edge <- -pl$from
inside <- main$placebo >= -edge & main$placebo < edge
stopifnot(identical(as.integer(table(cut(main$placebo[inside], seq(-edge, edge, by = 0.025), right = FALSE))),
                    as.integer(unlist(pl$counts))))
hist(main$placebo[inside], breaks = seq(-edge, edge, by = 0.025), main = "", xlab = "placebo slope (pp per 10 mm)")
abline(v = m$effect, col = "#c0392b", lwd = 2)

## Data and methods

- **Sources:** Die Bundeswahlleiterin, Wahlkreis results 2005–2025 and Wahlkreis shapefiles
  (2025 results: dl-de/by-2-0); Deutscher Wetterdienst, HYRAS-DE-PR v6-1 daily precipitation,
  CC BY 4.0. Reduced to `turnout.csv` and `rain.csv` with `rain_extract.py`.
- **Pipeline:** `rain_pipeline.R` (base R and `jsonlite`), console output in
  `rain-lowers-turnout.output.txt`.
- **Pitch:** every choice was fixed in `pitch.md` before any rain value or Wahlkreis result was
  read.
- **Licence of this notebook:** © John Brüne, see the README of pointzerofive-data.

In [ ]:
cat("All statistics reproduced ✓\n")